# CoreML Conversion + Parity Validation

Convert a pretrained ResNet-18 to Apple's CoreML format (`.mlpackage`), inspect what the format contains, then validate that the converted model produces the same predictions as the original PyTorch model.

**Pipeline:** PyTorch model → TorchScript (jit.trace) → `.mlpackage`

## Environment — WSL Required

`coremltools` only supports macOS and Linux — it won't install on Windows. This notebook runs on a Linux kernel via WSL2 (Ubuntu), using a shared `.venv-wsl` environment at the project root.

**To run:** Open in VS Code → kernel selector (top right) → "Select Another Kernel" → "Python Environments" → `.venv-wsl`.

## Cell 1 — Imports

In [2]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import coremltools as ct
import onnx
import onnxruntime as ort
import urllib.request
import os

print("torch:", torch.__version__)
print("coremltools:", ct.__version__)
print("onnx:", onnx.__version__)
print("onnxruntime:", ort.__version__)

torch: 2.12.0+cpu
coremltools: 9.0
onnx: 1.21.0
onnxruntime: 1.26.0


## Cell 2 — Load Model

`eval()` switches BatchNorm to use running statistics (accumulated during training) instead of batch statistics, and disables Dropout. Both are required for correct, deterministic inference — and for tracing to produce a stable graph.

In [3]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.eval()

num_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-18 loaded: {num_params:,} parameters")

ResNet-18 loaded: 11,689,512 parameters


## Cell 3 — Trace to TorchScript

`jit.trace` runs a forward pass with a dummy input and records every op that fires. The result is a frozen compute graph — no Python, no dynamic dispatch. This is what `coremltools` will consume.

`no_grad` is used because tracing is inference — we don't need gradients, don't want the memory overhead of building a computation graph, and the trace itself captures the graph structure separately.

In [4]:
dummy_input = torch.zeros(1, 3, 224, 224)
print("Dummy input shape:", dummy_input.shape)

with torch.no_grad():
    traced_model = torch.jit.trace(model, dummy_input)

print("Type:", type(traced_model))
print("Trace complete")

Dummy input shape: torch.Size([1, 3, 224, 224])
Type: <class 'torch.jit._trace.TopLevelTracedModule'>
Trace complete


## Cell 4 — Convert to CoreML + Export to ONNX

`ct.convert()` takes the TorchScript graph and maps every op to CoreML's operator set.

`torch.onnx.export` runs a second trace and emits an ONNX protobuf. We export both from the same model so they stay in sync — ONNX will serve as the parity proxy on Linux.

We pull mean/std directly from the weights metadata — the values are already there, no hardcoding.

We use `ct.TensorType` here since we're on Linux (WSL). In production on macOS, you'd use `ct.ImageType` to bake normalization into the model:
```python
# macOS production syntax (coremltools 9.0)
ct.ImageType(
    name="input",
    shape=(1, 3, 224, 224),
    scale=1 / (255.0 * 0.226),
    bias=[-m/s for m, s in zip(mean, std)],
)
```
This lets the iOS caller pass a raw `UIImage` — no preprocessing needed on their side.

In [10]:
weights = models.ResNet18_Weights.DEFAULT
mean = list(weights.transforms().mean)
std  = list(weights.transforms().std)
print("mean:", mean)
print("std: ", std)

coreml_model = ct.convert(
    traced_model,
    inputs=[ct.TensorType(name="input", shape=(1, 3, 224, 224))],
    minimum_deployment_target=ct.target.iOS15,
)
print("CoreML conversion complete")
print("Type:", type(coreml_model))

torch.onnx.export(
    model,
    dummy_input,
    "resnet18.onnx",
    dynamo=False,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=17,
)
print("ONNX export complete")

mean: [0.485, 0.456, 0.406]
std:  [0.229, 0.224, 0.225]


Running MIL backend_mlprogram pipeline: 100%|██████████████████████████████████████████████████████████| 12/12 [00:00<00:00, 466.67 passes/s]
/tmp/ipykernel_3759/2102560127.py:15: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


CoreML conversion complete
Type: <class 'coremltools.models.model.MLModel'>
ONNX export complete


## Cell 5 — Save Models

`.mlpackage` is a directory bundle — spec, weights, and metadata as separate files. `resnet18.onnx` is a single flatbuffer protobuf file.

In [11]:
import subprocess

coreml_model.save("resnet18.mlpackage")
result = subprocess.run(["du", "-sh", "resnet18.mlpackage"], capture_output=True, text=True)
print("CoreML size:", result.stdout.strip())

onnx_size_mb = os.path.getsize("resnet18.onnx") / 1_000_000
print(f"ONNX size:   {onnx_size_mb:.1f} MB")

CoreML size: 23M	resnet18.mlpackage
ONNX size:   46.7 MB


## Cell 6 — Load Test Image + Preprocess

Both models receive the same preprocessed input. PyTorch gets a tensor, CoreML gets the same data as a numpy array.

The preprocessing pipeline is identical for both — `Resize → CenterCrop → ToTensor → Normalize`. `ToTensor` converts pixels from uint8 (0–255) to float (0.0–1.0), which is required before `Normalize` can apply mean/std correctly.

In [13]:
pil_image = Image.open("../resnet-features/data/Amy.jpg").convert("RGB")
print("Original image size:", pil_image.size)

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

torch_input = preprocess(pil_image).unsqueeze(0)   # (1, 3, 224, 224) torch tensor
coreml_input = torch_input.numpy()                  # same data, numpy array for CoreML

print("torch_input shape:", torch_input.shape)
print("coreml_input shape:", coreml_input.shape)
print("dtype:", coreml_input.dtype)

Original image size: (711, 1263)
torch_input shape: torch.Size([1, 3, 224, 224])
coreml_input shape: (1, 3, 224, 224)
dtype: float32


## Cell 7 — PyTorch Inference

Run the original PyTorch model on the test image. This is the ground truth we'll compare CoreML against.

In [14]:
labels = weights.meta["categories"]

with torch.no_grad():
    torch_logits = model(torch_input)

print("Logits shape:", torch_logits.shape)

torch_probs = torch.softmax(torch_logits, dim=1)
top5_probs, top5_idx = torch.topk(torch_probs, 5)

print("\nPyTorch top-5 predictions:")
for i in range(5):
    print(f"  {top5_probs[0][i]:.4f}  {labels[top5_idx[0][i]]}")

pytorch_logits_np = torch_logits.numpy()  # keep for parity comparison

Logits shape: torch.Size([1, 1000])

PyTorch top-5 predictions:
  0.5912  tabby
  0.1632  Egyptian cat
  0.0761  tiger cat
  0.0345  Persian cat
  0.0284  quilt


## Cell 8 — Structural Verification + ONNX Runtime Setup

`coreml_model.predict()` requires macOS — coremltools 9.0 removed the Linux inference backend. Instead we:
1. Inspect the CoreML spec to confirm inputs/outputs are wired correctly
2. Set up an ONNX Runtime session as a parity proxy — same weights, same ops, runs on Linux

**To run full CoreML parity on macOS:**
```python
coreml_out = coreml_model.predict({"input": coreml_input})
coreml_logits_np = list(coreml_out.values())[0]
max_diff = np.max(np.abs(pytorch_logits_np - coreml_logits_np))
print(f"Max diff: {max_diff:.6f}")  # expect < 1e-3
```

In [16]:
spec = coreml_model.get_spec()
print("CoreML spec version:", spec.specificationVersion)
print("Model type:", spec.WhichOneof("Type"))

print("\n--- CoreML Inputs ---")
for inp in spec.description.input:
    print(f"  name: {inp.name}  type: {inp.type.WhichOneof('Type')}")

print("\n--- CoreML Outputs ---")
for out in spec.description.output:
    print(f"  name: {out.name}  type: {out.type.WhichOneof('Type')}")

# ONNX Runtime session
ort_session = ort.InferenceSession("resnet18.onnx")
ort_input_name = ort_session.get_inputs()[0].name
ort_input = {ort_input_name: coreml_input}

print("\n--- ONNX Runtime ---")
print("Input name: ", ort_input_name)
print("Input shape:", ort_session.get_inputs()[0].shape)
print("Output name:", ort_session.get_outputs()[0].name)

CoreML spec version: 6
Model type: mlProgram

--- CoreML Inputs ---
  name: input  type: multiArrayType

--- CoreML Outputs ---
  name: var_362  type: multiArrayType

--- ONNX Runtime ---
Input name:  input
Input shape: ['batch_size', 3, 224, 224]
Output name: output


## Cell 9 — Parity Validation: PyTorch vs ONNX

Run the same preprocessed image through ONNX Runtime and compare raw logits against PyTorch. Both models share the same weights — any divergence is floating-point precision noise, not a conversion bug.

Max diff under `1e-3` = clean conversion. Max diff over `1e-2` = something went wrong (wrong input, wrong normalization, wrong output node).

In [18]:
ort_outputs = ort_session.run(None, ort_input)
ort_logits_np = ort_outputs[0]  # shape: (1, 1000)

max_diff  = np.max(np.abs(pytorch_logits_np - ort_logits_np))
mean_diff = np.mean(np.abs(pytorch_logits_np - ort_logits_np))

print(f"Max absolute diff:  {max_diff:.6f}")
print(f"Mean absolute diff: {mean_diff:.6f}")
print(f"Parity: {'PASS' if np.allclose(pytorch_logits_np, ort_logits_np, atol=1e-4, rtol=1e-4) else 'FAIL'}")

pytorch_top1 = labels[np.argmax(pytorch_logits_np)]
onnx_top1    = labels[np.argmax(ort_logits_np)]

print(f"\nPyTorch top-1: {pytorch_top1}")
print(f"ONNX    top-1: {onnx_top1}")
print(f"Top-1 match:   {pytorch_top1 == onnx_top1}")

Max absolute diff:  0.000007
Mean absolute diff: 0.000002
Parity: PASS

PyTorch top-1: tabby
ONNX    top-1: tabby
Top-1 match:   True
